In [11]:
import requests, json, re

from docx import Document
import pandas as pd


In [12]:
def ask_ollama(model, prompt):
    r = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt},
        stream=True
    )
    out = ""
    for line in r.iter_lines():
        if line:
            try:
                out += json.loads(line.decode()).get("response", "")
            except:
                pass
    return out

In [13]:
def load_docx_text(path):
    doc = Document(path)
    return "\n".join(p.text for p in doc.paragraphs)

In [14]:
def extract_bibliography(text):
    # Find the start of the references section
    pattern = r"(References|Bibliography)\s*[\n\r]+(.+)"
    match = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL)
    if not match:
        return []
    
    refs_block = match.group(2)

    # Split into individual references (very forgiving)
    refs = re.split(r"\n\s*\n", refs_block)
    refs = [r.strip() for r in refs if len(r.strip()) > 5]
    return refs



In [15]:
def parse_reference(model, ref_text):
    prompt = f"""
Extract the following reference into JSON with these fields:
- title
- journal
- authors
- details (year, volume, issue, pages)
- doi
- link

Reference:
{ref_text}

Return ONLY valid JSON.
"""
    raw = ask_ollama(model, prompt)

    # Try to parse JSON
    try:
        return json.loads(raw)
    except:
        return {"title": None, "journal": None, "authors": None,
                "details": None, "doi": None, "link": None}


In [16]:
def bibliography_to_dataframe(docx_path, model):
    text = load_docx_text(docx_path)
    refs = extract_bibliography(text)

    parsed = []
    for ref in refs:
        parsed.append(parse_reference(model, ref))

    return pd.DataFrame(parsed)

In [17]:
%%time
df1 = bibliography_to_dataframe("Introduction.docx", model="mistral")
# df2 = bibliography_to_dataframe("Introduction.docx", model="phi3")
# df3 = bibliography_to_dataframe("Introduction.docx", model="medllama2")
# df4 = bibliography_to_dataframe("Introduction.docx", model="meditron")

CPU times: total: 250 ms
Wall time: 1min 24s


In [18]:
df1

,title,journal,authors,details,doi,link
0,A Detailed Review of pH and its Applications,Journal of Pharmaceutical and Biopharmaceutica...,"[Tiwari, R., Mahalpure, G. S.]","{'year': 2025, 'volume': 6, 'issue': 2, 'pages...",10.25082/JPBR.2024.02.001,<https://doi.org/10.25082/JPBR.2024.02.001>
1,Over One-Year Long-Term Laboratory Tests of pH...,Sensors (Basel),"[Wiora A, Wiora J]","{'year': '2018', 'volume': '18', 'issue': '12'...",10.3390/s18124102,https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6...
2,Variability in measuring pH using litmus paper...,Eye (Lond),"[Monaghan MT, Brogan K, Lockington D, Rotchfor...","{'year': 2020, 'volume': 34, 'issue': 11, 'pag...",10.1038/s41433-019-0737-z,PMID: 31848458; PMCID: PMC7784915.
3,Machine learning-based pH quantification from ...,Chemistry Letters,"[Yuto Nakamura, Masahiro Nagao]","{'year': '2025', 'volume': '54', 'issue': '2',...",10.1093/chemle/upaf018,https://doi.org/10.1093/chemle/upaf018
4,None,None,None,None,None,None
5,CLIA Acceptance Limits for Proficiency Testing,Westgard Quality Control,[Westgard QC],"{'year': 2025, 'volume': '', 'issue': '', 'pag...",,https://westgard.com/clia-a-quality/quality-re...
6,None,None,None,None,None,None


In [19]:
df1.title

0         A Detailed Review of pH and its Applications
1    Over One-Year Long-Term Laboratory Tests of pH...
2    Variability in measuring pH using litmus paper...
3    Machine learning-based pH quantification from ...
4                                                 None
5       CLIA Acceptance Limits for Proficiency Testing
6                                                 None
Name: title, dtype: object